In [5]:
import duckdb
cn = duckdb.connect()

cn.execute("create view v_order_header as select * from 'data/invoice_header_jn.csv'")
cn.execute("create view v_order_line as select * from 'data/invoice_detail_jn.csv'")

In [11]:
# Simple timeline for invoice 100000
cn.sql("""
    select
        invoice_id,
        status_cd,
        invoice_total_amt,
        begin_ts,
        end_ts
    from v_order_header
    where invoice_id = 100000
    order by begin_ts
""").show()

# As-of snapshot for invoice 100000
as_of_ts = '2026-01-01 12:00:01'

sql = """
    select
        h.invoice_id,
        h.status_cd,
        h.invoice_total_amt,
        h.begin_ts as header_begin_ts,
        h.end_ts as header_end_ts,
        d.line_nbr,
        d.item_cd,
        d.qty,
        d.unit_price,
        d.line_amt
    from v_order_header h
    left join v_order_line d
      on d.invoice_id = h.invoice_id
    where h.invoice_id = ?
      and cast(? as timestamp) between h.begin_ts and h.end_ts
    order by d.line_nbr
"""

cn.sql(sql, params=[100000, as_of_ts]).show()

┌────────────┬───────────┬───────────────────┬─────────────────────┬─────────────────────┐
│ invoice_id │ status_cd │ invoice_total_amt │      begin_ts       │       end_ts        │
│   int64    │  varchar  │      double       │      timestamp      │      timestamp      │
├────────────┼───────────┼───────────────────┼─────────────────────┼─────────────────────┤
│     100000 │ DRAFT     │            163.15 │ 2026-01-01 08:00:00 │ 2026-01-01 12:00:00 │
│     100000 │ SUBMITTED │            163.15 │ 2026-01-01 12:00:01 │ 2026-01-01 18:00:01 │
│     100000 │ APPROVED  │            163.76 │ 2026-01-01 18:00:02 │ 2026-01-01 23:00:02 │
│     100000 │ POSTED    │            163.76 │ 2026-01-01 23:00:03 │ 2026-01-02 23:00:03 │
│     100000 │ CANCELED  │            163.76 │ 2026-01-02 23:00:04 │ 9999-12-31 23:59:59 │
└────────────┴───────────┴───────────────────┴─────────────────────┴─────────────────────┘

┌────────────┬───────────┬───────────────────┬─────────────────────┬─────────────────────